# TRELLIS.2 — Multi-View vs Single-Image Demo (Colab A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TylerOlszewski/TRELLIS.2/blob/main/notebooks/TRELLIS2_MultiImage_Colab_A100.ipynb)

Generates two **textured 3D assets (GLB + turntable video)** from the same object for a
direct demo comparison:

1. A **multi-view run** using any number of views (four well-spaced views are recommended).
2. A **single-image run** using the held-out straight-on front image.

The multi-view demo uses `Trellis2ImageTo3DPipeline.run_multi_image()`; the single-image
demo uses the pipeline's native `run()` method. Camera poses are not required.

> **Damaged open-door Camry:** run setup cells 1–7 and pipeline cell 9 (skip Demo 1's
> upload cell 8), then jump to the dedicated car section
> after Demo 2. It accepts a prepared multi-view ZIP, compares several 512-resolution
> geometry seeds, and spends the high-resolution pass only on the selected seed.

**Requirements**
- Colab **A100 GPU** and the **2025.10 runtime** (Runtime → Change runtime type).
  Pinning the runtime gives the notebook Python 3.12 + PyTorch 2.8 and keeps the compiled
  extension ABI reproducible as Colab's default image changes.
- A Hugging Face account with access to two **gated** models (see the auth cell below).
- Google Drive with a few GB free (used to cache compiled CUDA wheels between sessions).

**Timing**: the first run downloads a prebuilt FlashAttention wheel and compiles the other
CUDA extensions (typically ~15–40 min). Wheels are cached to Drive, so later sessions set
up in a few minutes. Model download and generation time are additional.

Run the cells top to bottom. Setup is idempotent: reconnecting or rerunning a cell reuses
every successfully cached wheel. The default seeds and resolution match between both runs
so the comparison changes only the image conditioning.

In [ ]:
#@title 1. Verify the pinned A100 runtime
!nvidia-smi
import sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab choose Runtime → Change runtime type → A100 GPU.')
name = torch.cuda.get_device_name(0)
runtime_ok = sys.version_info[:2] == (3, 12) and torch.__version__.split('+')[0].startswith('2.8.')
print(f"\nPython {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {name}")
if 'A100' not in name:
    raise RuntimeError(f'Expected an A100, but Colab assigned {name}. Change the hardware accelerator to A100.')
if not runtime_ok:
    raise RuntimeError(
        'Select Runtime → Change runtime type → Runtime Version 2025.10, then reconnect. '
        'This notebook intentionally pins Python 3.12 / PyTorch 2.8 for binary compatibility.'
    )
print('✓ compatible Colab A100 runtime')

In [ ]:
#@title 2. Mount Google Drive & set up caches
USE_DRIVE = True  #@param {type:"boolean"}
CACHE_MODELS_ON_DRIVE = False  #@param {type:"boolean"}

# Wheels (slow to build, small) always go to Drive when USE_DRIVE is on.
# Model checkpoints (~15 GB, fast to re-download) only go to Drive if you have the space.
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/TRELLIS2_cache'
else:
    CACHE_ROOT = '/content/TRELLIS2_cache'

WHEELS_DIR = f'{CACHE_ROOT}/wheels'
HF_HOME = f'{CACHE_ROOT}/hf_home' if (USE_DRIVE and CACHE_MODELS_ON_DRIVE) else '/content/hf_home'
os.makedirs(WHEELS_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
print('wheel cache :', WHEELS_DIR)
print('HF cache    :', HF_HOME)

In [ ]:
#@title 3. Prepare the CUDA build environment and wheel cache
# The pinned 2025.10 runtime has PyTorch/CUDA 12.6 and nvcc 12.5. PyTorch accepts a
# same-major minor-version difference when compiling extensions; a major mismatch is unsafe.
import os, re, shutil, subprocess, sys, pathlib, torch

def _versions():
    nvcc_path = shutil.which('nvcc')
    if nvcc_path is None:
        raise RuntimeError('nvcc is missing from this runtime; select Colab runtime 2025.10.')
    nvcc = subprocess.run([nvcc_path, '--version'], capture_output=True, text=True, check=True).stdout
    m = re.search(r'release (\d+)\.(\d+)', nvcc)
    if m is None or torch.version.cuda is None:
        raise RuntimeError('Could not determine the CUDA toolchain versions.')
    return nvcc_path, (int(m.group(1)), int(m.group(2))), tuple(int(x) for x in torch.version.cuda.split('.')[:2])

nvcc_path, (nv_maj, nv_min), (t_maj, t_min) = _versions()
print(f"system nvcc: {nv_maj}.{nv_min} | torch built for CUDA: {t_maj}.{t_min}")

if nv_maj != t_maj:
    raise RuntimeError(
        f'nvcc {nv_maj}.{nv_min} and torch CUDA {t_maj}.{t_min} have different major versions. '
        'Reconnect using Colab runtime 2025.10 instead of compiling an incompatible extension.'
    )
if nv_min != t_min:
    print('ℹ same CUDA major version; the 12.5/12.6 minor difference is expected on this runtime')

# CUDA_HOME must point at the toolkit that owns the active nvcc.
os.environ['CUDA_HOME'] = str(pathlib.Path(nvcc_path).resolve().parents[1])
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'  # compile only for the A100 architecture
os.environ['MAX_JOBS'] = str(min(8, os.cpu_count() or 2))

# Invalidate cached wheels when the torch/CUDA/python environment changed — wheels
# compiled against a different torch ABI crash at import time.
abi = int(torch.compiled_with_cxx11_abi())
env_tag = (f"torch{torch.__version__}-cu{torch.version.cuda}-nvcc{nv_maj}.{nv_min}-"
           f"py{sys.version_info.major}.{sys.version_info.minor}-abi{abi}-sm80")
tag_file = pathlib.Path(WHEELS_DIR) / 'env_tag.txt'
if tag_file.exists() and tag_file.read_text().strip() != env_tag:
    print(f"environment changed ({tag_file.read_text().strip()} → {env_tag}); clearing cached wheels…")
    for whl in pathlib.Path(WHEELS_DIR).glob('*.whl'):
        whl.unlink()
tag_file.write_text(env_tag)
print("wheel-cache tag:", env_tag)

In [ ]:
#@title 4. Clone repo & install Python dependencies
import os, subprocess, sys
REPO_URL = 'https://github.com/TylerOlszewski/TRELLIS.2.git'
REPO_REF = 'codex/colab-multi-image'  # Use 'main' after this PR is merged.
if not os.path.isdir('/content/TRELLIS.2'):
    !git clone --branch {REPO_REF} --single-branch {REPO_URL} /content/TRELLIS.2
else:
    !git -C /content/TRELLIS.2 fetch origin {REPO_REF}
    !git -C /content/TRELLIS.2 checkout --detach FETCH_HEAD
%cd /content/TRELLIS.2

deps = [
    'imageio', 'imageio-ffmpeg', 'tqdm', 'easydict', 'opencv-python-headless',
    'ninja', 'trimesh', 'kornia', 'timm', 'packaging', 'psutil', 'wheel',
    'plyfile', 'zstandard', 'matplotlib', 'huggingface_hub', 'pillow-heif',
    'transformers>=4.56.0,<5',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],
               check=True)
print('✓ python deps installed')

In [ ]:
#@title 5. Hugging Face login (gated models)
# Two models used by the pipeline are GATED on Hugging Face — request access first
# (accept the license on each model page before running this cell):
#   • https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m   (image encoder)
#   • https://huggingface.co/briaai/RMBG-2.0                            (background removal)
# Then create a *read* token at https://huggingface.co/settings/tokens and add it to
# Colab Secrets (key icon in the left sidebar) under the name HF_TOKEN.
# This cell verifies both gated repositories before the expensive pipeline download.
import os
from huggingface_hub import get_token, hf_hub_download, login, whoami
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token  # TRELLIS passes this directly to Transformers.
    login(token=token)
    print('✓ logged in with Colab secret HF_TOKEN')
else:
    print('No HF_TOKEN Colab secret found — falling back to interactive login:')
    login()
    token = get_token()

if not token:
    raise RuntimeError('Hugging Face login did not provide a token.')
os.environ['HF_TOKEN'] = token
print('Hugging Face account:', whoami(token=token)['name'])
for repo_id in ('facebook/dinov3-vitl16-pretrain-lvd1689m', 'briaai/RMBG-2.0'):
    hf_hub_download(repo_id, 'config.json', token=token)
    print('✓ gated-model access verified:', repo_id)

In [ ]:
#@title 6. Build / install CUDA extensions (cached on Drive after first run)
# FlashAttention uses an official prebuilt wheel. The five source-built wheels are pinned
# to immutable revisions and cached as each one succeeds. Rerunning this cell is safe.
import glob, os, shutil, subprocess, sys, torch, pathlib

EXT_ROOT = '/tmp/trellis2_extensions'
os.makedirs(EXT_ROOT, exist_ok=True)

# O-Voxel vendors Eigen as a header-only dependency, but the repository checkout
# intentionally leaves that directory empty. Install the distro headers once and
# expose them at the include path used by o-voxel/setup.py.
eigen_headers = pathlib.Path('/usr/include/eigen3/Eigen')
if not eigen_headers.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libeigen3-dev'], check=True)
eigen_link = pathlib.Path('/content/TRELLIS.2/o-voxel/third_party/eigen/Eigen')
if not eigen_link.exists():
    eigen_link.symlink_to(eigen_headers, target_is_directory=True)
print('✓ Eigen headers ready for o-voxel')

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

def _clone(url, name, ref):
    dst = os.path.join(EXT_ROOT, name)
    if os.path.exists(dst) and not os.path.isdir(os.path.join(dst, '.git')):
        shutil.rmtree(dst)  # discard only an incomplete clone in the ephemeral /tmp tree
    if not os.path.exists(dst):
        subprocess.run(['git', 'clone', '--recursive', url, dst], check=True)
    subprocess.run(['git', '-C', dst, 'checkout', '--detach', ref], check=True)
    subprocess.run(['git', '-C', dst, 'submodule', 'update', '--init', '--recursive'], check=True)
    return dst

def _wheel(src):
    _pip('wheel', '--no-build-isolation', '--no-deps', '-w', WHEELS_DIR, src)

def _download(url):
    _pip('download', '--no-deps', '-d', WHEELS_DIR, url)

def ensure(import_name, wheel_prefix, build):
    try:
        __import__(import_name)
        print(f'✓ {import_name} already working')
        return
    except Exception:
        pass
    cached = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    if cached:
        print(f'→ installing {import_name} from cached wheel: {os.path.basename(cached[-1])}')
        _pip('install', '-q', '--force-reinstall', '--no-deps', cached[-1])
        try:
            __import__(import_name)
            print(f'✓ {import_name} (from cache)')
            return
        except Exception as e:
            print(f'  cached wheel unusable ({type(e).__name__}) — replacing it…')
            for path in cached:
                os.unlink(path)
    print(f'→ preparing {import_name} (source builds can take a while)…')
    build()
    built = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    assert built, f'build produced no wheel matching {wheel_prefix}-*.whl in {WHEELS_DIR}'
    _pip('install', '-q', '--force-reinstall', '--no-deps', built[-1])
    __import__(import_name)
    print(f'✓ {import_name} (installed and cached)')

ABI_LABEL = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
FLASH_WHEEL = (
    'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/'
    f'flash_attn-2.8.3+cu12torch2.8cxx11abi{ABI_LABEL}-cp312-cp312-linux_x86_64.whl'
)
ensure('flash_attn', 'flash_attn',
       lambda: _download(FLASH_WHEEL))
ensure('nvdiffrast', 'nvdiffrast',
       lambda: _wheel(_clone('https://github.com/NVlabs/nvdiffrast.git', 'nvdiffrast',
                                  '253ac4fcea7de5f396371124af597e6cc957bfae')))
ensure('nvdiffrec_render', 'nvdiffrec_render',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/nvdiffrec.git', 'nvdiffrec',
                                  'b296927cc7fd01c2ac1087c8065c4d7248f72da4')))
ensure('cumesh', 'cumesh',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/CuMesh.git', 'CuMesh',
                                  '12289e1062f0603f2f0d0771b02e1395d247f26f')))
ensure('flex_gemm', 'flex_gemm',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/FlexGEMM.git', 'FlexGEMM',
                                  '6dd94a859c26ee8246888502eada3dd8ad85532e')))
ensure('o_voxel', 'o_voxel',
       lambda: _wheel('/content/TRELLIS.2/o-voxel'))

print('\n✓ all CUDA extensions ready')

In [ ]:
#@title 7. Sanity check
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content/TRELLIS.2
import o_voxel
from pillow_heif import register_heif_opener
register_heif_opener()
from trellis2.pipelines import Trellis2ImageTo3DPipeline  # prints the sparse backends line
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
print('✓ TRELLIS.2 imports OK (including HEIC/HEIF image support)')

## Demo 1 — Any number of input views

Upload one or more views of the same object. Four well-spaced views are the recommended
starting point, for example:

`front-left → back-left → back-right → front-right`

If you want the controlled comparison in Demo 2, reserve the straight-on front image for it.
Otherwise, any useful angles can be included. No camera poses are needed. Use the same
object state, framing, and lighting throughout, and number filenames in orbit order.
PNG, JPG/JPEG, WebP, HEIC, and HEIF images are accepted directly.

In [ ]:
#@title 8. Upload or select any number of views
MULTIVIEW_IMAGE_SOURCE = 'upload'  #@param ["upload", "drive_folder"]
MULTIVIEW_DRIVE_FOLDER = '/content/drive/MyDrive/TRELLIS2_inputs/multiview'  #@param {type:"string"}

import math, os, shutil
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp', '.heic', '.heif')
MULTIVIEW_INPUT_DIR = '/content/input_views/multiview'
shutil.rmtree(MULTIVIEW_INPUT_DIR, ignore_errors=True)
os.makedirs(MULTIVIEW_INPUT_DIR)

if MULTIVIEW_IMAGE_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            with open(os.path.join(MULTIVIEW_INPUT_DIR, os.path.basename(name)), 'wb') as f:
                f.write(data)
        else:
            print(f'Ignoring unsupported file: {name}')
else:
    if not os.path.isdir(MULTIVIEW_DRIVE_FOLDER):
        raise FileNotFoundError(f'Drive input folder does not exist: {MULTIVIEW_DRIVE_FOLDER}')
    for name in sorted(os.listdir(MULTIVIEW_DRIVE_FOLDER)):
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            shutil.copy(os.path.join(MULTIVIEW_DRIVE_FOLDER, name), MULTIVIEW_INPUT_DIR)

MULTIVIEW_IMAGE_PATHS = sorted(
    os.path.join(MULTIVIEW_INPUT_DIR, f) for f in os.listdir(MULTIVIEW_INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
)
if not MULTIVIEW_IMAGE_PATHS:
    raise ValueError('No supported images found. Upload at least one PNG, JPG/JPEG, WebP, HEIC, or HEIF file.')
view_count = len(MULTIVIEW_IMAGE_PATHS)
print(f'✓ {view_count} view(s) loaded from {MULTIVIEW_INPUT_DIR}')
if view_count == 1:
    print('Warning: one view is allowed, but it is equivalent to single-image conditioning.')

from PIL import Image
for path in MULTIVIEW_IMAGE_PATHS:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        raise ValueError(f'Unreadable image {os.path.basename(path)}: {exc}') from exc

import matplotlib.pyplot as plt
preview_columns = min(4, view_count)
preview_rows = math.ceil(view_count / preview_columns)
fig, axes = plt.subplots(
    preview_rows, preview_columns, figsize=(4 * preview_columns, 3.5 * preview_rows), squeeze=False
)
flat_axes = axes.ravel()
for ax, p in zip(flat_axes, MULTIVIEW_IMAGE_PATHS):
    with Image.open(p) as image:
        ax.imshow(image.convert('RGBA'))
    ax.set_title(os.path.basename(p), fontsize=9)
    ax.axis('off')
for ax in flat_axes[view_count:]:
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 9. Load the TRELLIS.2-4B pipeline (~15 GB download on first run)
from trellis2.pipelines import Trellis2ImageTo3DPipeline
if 'pipeline' not in globals():
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B')
    pipeline.cuda()  # low-VRAM mode: submodels move to GPU only while in use
print('✓ pipeline ready')

In [ ]:
#@title 10. Generate the multi-view 3D asset
MODE = 'stochastic'  #@param ["stochastic", "multidiffusion"]
RESOLUTION = 'default'  #@param ["default", "512", "1024", "1024_cascade", "1536_cascade"]
SEED = 42  #@param {type:"integer"}
PREPROCESS = True  #@param {type:"boolean"}
MULTIVIEW_OUTPUT_NAME = 'trellis2_multiview'  #@param {type:"string"}
# MODE:       'stochastic' cycles through views across denoising steps (fast);
#             'multidiffusion' averages all views at every step (slower, more stable).
# RESOLUTION: 'default' uses the model's config (1024_cascade). Use '512' if you hit OOM.
# PREPROCESS: automatic background removal + recentering. Disable only for clean-alpha inputs.

import os, torch
assert MULTIVIEW_OUTPUT_NAME and os.path.basename(MULTIVIEW_OUTPUT_NAME) == MULTIVIEW_OUTPUT_NAME, (
    'MULTIVIEW_OUTPUT_NAME must be a plain filename.'
)
from PIL import Image, ImageOps

multiview_images = []
for path in MULTIVIEW_IMAGE_PATHS:
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        mode = 'RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB'
        multiview_images.append(image.convert(mode).copy())
multiview_mesh = pipeline.run_multi_image(
    multiview_images,
    seed=SEED,
    mode=MODE,
    pipeline_type=None if RESOLUTION == 'default' else RESOLUTION,
    preprocess_image=PREPROCESS,
)[0]
multiview_mesh.simplify(16777216)  # nvdiffrast limit
torch.cuda.empty_cache()
print(
    f'✓ {len(multiview_images)}-view mesh generated: {multiview_mesh.vertices.shape[0]:,} vertices, '
    f'{multiview_mesh.faces.shape[0]:,} faces'
)

In [ ]:
#@title 11. Render the multi-view turntable video
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Cell 12 parks the mesh on CPU to free VRAM for Demo 2; rendering needs it back on the GPU.
if multiview_mesh.device.type != 'cuda':
    multiview_mesh = multiview_mesh.cuda()

HDRI_PATH = 'assets/hdri/forest.exr'
hdri = cv2.imread(HDRI_PATH, cv2.IMREAD_UNCHANGED)
if hdri is None:
    raise FileNotFoundError(
        f'Could not read {HDRI_PATH}. Re-run cell 7 so the working directory is '
        '/content/TRELLIS.2 and OpenEXR support is enabled.'
    )
envmap = EnvMap(torch.tensor(
    cv2.cvtColor(hdri, cv2.COLOR_BGR2RGB), dtype=torch.float32, device='cuda'
))
multiview_video = render_utils.make_pbr_vis_frames(
    render_utils.render_video(multiview_mesh, envmap=envmap)
)
multiview_video_path = f'{OUT_DIR}/{MULTIVIEW_OUTPUT_NAME}.mp4'
imageio.mimsave(multiview_video_path, multiview_video, fps=15)

from IPython.display import Video, display
display(Video(multiview_video_path, embed=True, width=512))

In [ ]:
#@title 12. Export the multi-view GLB (and copy results to Drive)
EXPORT_REMESH = False  #@param {type:"boolean"}
REMESH_BAND = 1.0  #@param {type:"number"}
REMESH_PROJECT = 0.9  #@param {type:"number"}
# EXPORT_REMESH=False uses O-Voxel's duplicate/non-manifold/component cleanup path.
# Turn it on only when you specifically want narrow-band dual-contouring remeshing.

import gc, os, shutil, torch
import o_voxel

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# to_glb runs CUDA kernels (nvdiffrast / cumesh / flex_gemm). The teardown at the end of
# this cell parks the mesh on CPU, so move it back when the cell is re-run.
if multiview_mesh.device.type != 'cuda':
    multiview_mesh = multiview_mesh.cuda()

glb = o_voxel.postprocess.to_glb(
    vertices          = multiview_mesh.vertices,
    faces             = multiview_mesh.faces,
    attr_volume       = multiview_mesh.attrs,
    coords            = multiview_mesh.coords,
    attr_layout       = multiview_mesh.layout,
    voxel_size        = multiview_mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = 1000000,
    texture_size      = 4096,
    remesh            = EXPORT_REMESH,
    remesh_band       = REMESH_BAND,
    remesh_project    = REMESH_PROJECT,
    verbose           = True,
)
multiview_glb_path = f'{OUT_DIR}/{MULTIVIEW_OUTPUT_NAME}.glb'
glb.export(multiview_glb_path)
print('✓ exported', multiview_glb_path)

if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(multiview_glb_path, drive_out)
    if 'multiview_video_path' in globals() and os.path.exists(multiview_video_path):
        shutil.copy(multiview_video_path, drive_out)
    print('✓ copied results to', drive_out)

from google.colab import files as colab_files
colab_files.download(multiview_glb_path)

# The GLB is on disk now. Park the mesh on CPU and drop the render frames so Demo 2 starts
# with a clean GPU; re-running cell 11 or 12 moves the mesh back automatically.
multiview_mesh = multiview_mesh.cpu()
for _name in ('multiview_video', 'glb'):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## Demo 2 — Single straight-on front image

Use the held-out front image of the same object. Keeping the seed, resolution, and
preprocessing settings matched to Demo 1 makes the outputs easier to compare.

In [ ]:
#@title 13. Upload or select the held-out front image
SINGLE_IMAGE_SOURCE = 'upload'  #@param ["upload", "drive_file"]
SINGLE_IMAGE_DRIVE_PATH = '/content/drive/MyDrive/TRELLIS2_inputs/front.png'  #@param {type:"string"}

import os, shutil
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp', '.heic', '.heif')
SINGLE_INPUT_DIR = '/content/input_views/single_front'
shutil.rmtree(SINGLE_INPUT_DIR, ignore_errors=True)
os.makedirs(SINGLE_INPUT_DIR)

if SINGLE_IMAGE_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    valid_uploads = [
        (name, data) for name, data in uploaded.items()
        if name.lower().endswith(SUPPORTED_EXTENSIONS)
    ]
    if len(valid_uploads) != 1:
        found = ', '.join(name for name, _ in valid_uploads) or '(none)'
        raise ValueError(
            f'Expected exactly 1 supported image, found {len(valid_uploads)}: {found}. '
            'Upload only the straight-on front view and re-run this cell.'
        )
    name, data = valid_uploads[0]
    SINGLE_IMAGE_PATH = os.path.join(SINGLE_INPUT_DIR, os.path.basename(name))
    with open(SINGLE_IMAGE_PATH, 'wb') as f:
        f.write(data)
else:
    if not os.path.isfile(SINGLE_IMAGE_DRIVE_PATH):
        raise FileNotFoundError(f'Drive input image does not exist: {SINGLE_IMAGE_DRIVE_PATH}')
    if not SINGLE_IMAGE_DRIVE_PATH.lower().endswith(SUPPORTED_EXTENSIONS):
        raise ValueError(f'Unsupported image type: {SINGLE_IMAGE_DRIVE_PATH}')
    SINGLE_IMAGE_PATH = os.path.join(SINGLE_INPUT_DIR, os.path.basename(SINGLE_IMAGE_DRIVE_PATH))
    shutil.copy(SINGLE_IMAGE_DRIVE_PATH, SINGLE_IMAGE_PATH)

from PIL import Image
from IPython.display import display
try:
    with Image.open(SINGLE_IMAGE_PATH) as image:
        image.verify()
except Exception as exc:
    raise ValueError(f'Unreadable image {os.path.basename(SINGLE_IMAGE_PATH)}: {exc}') from exc
with Image.open(SINGLE_IMAGE_PATH) as image:
    single_preview = image.convert('RGBA')
print('✓ held-out front image loaded:', os.path.basename(SINGLE_IMAGE_PATH))
display(single_preview)

In [ ]:
#@title 14. Generate the single-image 3D asset
SINGLE_RESOLUTION = 'default'  #@param ["default", "512", "1024", "1024_cascade", "1536_cascade"]
SINGLE_SEED = 42  #@param {type:"integer"}
SINGLE_PREPROCESS = True  #@param {type:"boolean"}
SINGLE_OUTPUT_NAME = 'trellis2_single_front'  #@param {type:"string"}

import os, torch
from PIL import Image, ImageOps
assert SINGLE_OUTPUT_NAME and os.path.basename(SINGLE_OUTPUT_NAME) == SINGLE_OUTPUT_NAME, (
    'SINGLE_OUTPUT_NAME must be a plain filename.'
)

with Image.open(SINGLE_IMAGE_PATH) as image:
    image = ImageOps.exif_transpose(image)
    mode = 'RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB'
    single_image = image.convert(mode).copy()
single_mesh = pipeline.run(
    single_image,
    seed=SINGLE_SEED,
    pipeline_type=None if SINGLE_RESOLUTION == 'default' else SINGLE_RESOLUTION,
    preprocess_image=SINGLE_PREPROCESS,
)[0]
single_mesh.simplify(16777216)  # nvdiffrast limit
torch.cuda.empty_cache()
print(
    f'✓ single-image mesh generated: {single_mesh.vertices.shape[0]:,} vertices, '
    f'{single_mesh.faces.shape[0]:,} faces'
)

In [ ]:
#@title 15. Render the single-image turntable video
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
from IPython.display import Video, display

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

if single_mesh.device.type != 'cuda':
    single_mesh = single_mesh.cuda()

if 'envmap' not in globals():  # Demo 2 can be rendered without having run cell 11.
    HDRI_PATH = 'assets/hdri/forest.exr'
    hdri = cv2.imread(HDRI_PATH, cv2.IMREAD_UNCHANGED)
    if hdri is None:
        raise FileNotFoundError(
            f'Could not read {HDRI_PATH}. Re-run cell 7 so the working directory is '
            '/content/TRELLIS.2 and OpenEXR support is enabled.'
        )
    envmap = EnvMap(torch.tensor(
        cv2.cvtColor(hdri, cv2.COLOR_BGR2RGB), dtype=torch.float32, device='cuda'
    ))

single_video = render_utils.make_pbr_vis_frames(
    render_utils.render_video(single_mesh, envmap=envmap)
)
single_video_path = f'{OUT_DIR}/{SINGLE_OUTPUT_NAME}.mp4'
imageio.mimsave(single_video_path, single_video, fps=15)
display(Video(single_video_path, embed=True, width=512))

In [ ]:
#@title 16. Export the single-image GLB (and copy results to Drive)
SINGLE_EXPORT_REMESH = False  #@param {type:"boolean"}
SINGLE_REMESH_BAND = 1.0  #@param {type:"number"}
SINGLE_REMESH_PROJECT = 0.9  #@param {type:"number"}

import gc, os, shutil, torch
import o_voxel

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

if single_mesh.device.type != 'cuda':
    single_mesh = single_mesh.cuda()

single_glb = o_voxel.postprocess.to_glb(
    vertices          = single_mesh.vertices,
    faces             = single_mesh.faces,
    attr_volume       = single_mesh.attrs,
    coords            = single_mesh.coords,
    attr_layout       = single_mesh.layout,
    voxel_size        = single_mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = 1000000,
    texture_size      = 4096,
    remesh            = SINGLE_EXPORT_REMESH,
    remesh_band       = SINGLE_REMESH_BAND,
    remesh_project    = SINGLE_REMESH_PROJECT,
    verbose           = True,
)
single_glb_path = f'{OUT_DIR}/{SINGLE_OUTPUT_NAME}.glb'
single_glb.export(single_glb_path)
print('✓ exported', single_glb_path)

if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(single_glb_path, drive_out)
    if 'single_video_path' in globals() and os.path.exists(single_video_path):
        shutil.copy(single_video_path, drive_out)
    print('✓ copied results to', drive_out)

from google.colab import files as colab_files
colab_files.download(single_glb_path)

single_mesh = single_mesh.cpu()
globals().pop('single_video', None)
gc.collect()
torch.cuda.empty_cache()

print('\nComparison ready:')
if 'multiview_glb_path' in globals():
    print(f'  multi-view ({len(MULTIVIEW_IMAGE_PATHS)}):', multiview_glb_path)
print('  single     :', single_glb_path)

## Dedicated workflow — damaged Camry with both front doors open

This path is tuned for prepared car cutouts and accepts any positive number of views.
Unlike Demo 1, it can intentionally include the straight-front view because the doors,
missing rear glass, and collision
damage make that view useful geometry evidence. Upload the prepared
`open_door_pics_trellis_ready.zip`; its numbered filenames define the orbit, so browser
upload order does not matter.

Run setup cells **1–7**, skip Demo 1's upload cell 8, run pipeline cell **9**, then run
cells **17–21** below. Cell 18 makes inexpensive 512
candidates and ranks obvious fragmentation automatically. Check its gallery and override
the chosen seed in cell 19 if your eye prefers another candidate.


In [ ]:
#@title 17. Load a prepared Camry multi-view package
CAR_IMAGE_SOURCE = 'upload_zip'  #@param ["upload_zip", "drive_folder"]
CAR_DRIVE_FOLDER = '/content/drive/MyDrive/TRELLIS2_inputs/open_door_camry'  #@param {type:"string"}

import math, os, shutil, zipfile
from PIL import Image

CAR_INPUT_DIR = '/content/input_views/open_door_camry'
shutil.rmtree(CAR_INPUT_DIR, ignore_errors=True)
os.makedirs(CAR_INPUT_DIR)
CAR_SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp', '.heic', '.heif')

if CAR_IMAGE_SOURCE == 'upload_zip':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one file: open_door_pics_trellis_ready.zip')
    upload_name, upload_data = next(iter(uploaded.items()))
    if not upload_name.lower().endswith('.zip'):
        raise ValueError('The car workflow expects the prepared .zip package.')
    zip_path = os.path.join('/content', os.path.basename(upload_name))
    with open(zip_path, 'wb') as f:
        f.write(upload_data)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            name = os.path.basename(member.filename)
            if not name.lower().endswith(CAR_SUPPORTED_EXTENSIONS):
                continue
            destination = os.path.join(CAR_INPUT_DIR, name)
            if os.path.exists(destination):
                raise ValueError(f'Duplicate image filename in ZIP: {name}')
            with archive.open(member) as src, open(destination, 'wb') as dst:
                shutil.copyfileobj(src, dst)
else:
    if not os.path.isdir(CAR_DRIVE_FOLDER):
        raise FileNotFoundError(f'Drive input folder does not exist: {CAR_DRIVE_FOLDER}')
    for name in sorted(os.listdir(CAR_DRIVE_FOLDER)):
        src = os.path.join(CAR_DRIVE_FOLDER, name)
        if os.path.isfile(src) and name.lower().endswith(CAR_SUPPORTED_EXTENSIONS):
            shutil.copy(src, CAR_INPUT_DIR)

found = sorted(
    name for name in os.listdir(CAR_INPUT_DIR)
    if name.lower().endswith(CAR_SUPPORTED_EXTENSIONS)
)
if not found:
    raise ValueError('The car package contains no supported images.')
CAR_IMAGE_PATHS = [os.path.join(CAR_INPUT_DIR, name) for name in found]
car_images = []
for path in CAR_IMAGE_PATHS:
    with Image.open(path) as image:
        if image.size != (1024, 1024):
            raise ValueError(f'{os.path.basename(path)} must remain 1024 x 1024.')
        car_images.append(image.convert('RGB').copy())

import matplotlib.pyplot as plt
car_view_count = len(car_images)
car_preview_columns = min(4, car_view_count)
car_preview_rows = math.ceil(car_view_count / car_preview_columns)
fig, axes = plt.subplots(
    car_preview_rows, car_preview_columns,
    figsize=(4 * car_preview_columns, 3.5 * car_preview_rows), squeeze=False,
)
flat_axes = axes.ravel()
for ax, image, name in zip(flat_axes, car_images, found):
    ax.imshow(image)
    ax.set_title(name, fontsize=9)
    ax.axis('off')
for ax in flat_axes[car_view_count:]:
    ax.axis('off')
plt.tight_layout(); plt.show()
print(
    f'✓ prepared {car_view_count}-view car orbit loaded; '
    'PREPROCESS=False will preserve its shared scale and padding'
)

In [ ]:
#@title 18. Compare low-cost geometry candidates
CAR_CANDIDATE_SEEDS = '42, 123, 2026, 31415'  #@param {type:"string"}
CAR_CANDIDATE_MODE = 'stochastic'  #@param ["stochastic", "multidiffusion"]

import gc, cv2, numpy as np, torch
import matplotlib.pyplot as plt
from trellis2.utils import render_utils

candidate_seeds = [int(v.strip()) for v in CAR_CANDIDATE_SEEDS.split(',') if v.strip()]
if not candidate_seeds:
    raise ValueError('Enter at least one integer seed.')
if len(candidate_seeds) > 8:
    raise ValueError('Use at most eight candidates in one pass.')

def _silhouette_score(alpha_frames):
    continuity, areas = [], []
    for frame in alpha_frames:
        mask = np.max(frame, axis=2) > 12
        areas.append(float(mask.mean()))
        count, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
        component_areas = stats[1:, cv2.CC_STAT_AREA] if count > 1 else np.array([0])
        continuity.append(float(component_areas.max() / max(mask.sum(), 1)))
    # Prefer one connected silhouette from every angle and penalize unstable scale.
    return float(np.mean(continuity) - 0.10 * np.std(areas) / max(np.mean(areas), 1e-6))

car_candidate_results = []
for seed in candidate_seeds:
    print(f'\n=== Candidate seed {seed} ===')
    candidate_mesh = pipeline.run_multi_image(
        car_images,
        seed=seed,
        mode=CAR_CANDIDATE_MODE,
        pipeline_type='512',
        preprocess_image=False,
    )[0]
    candidate_mesh.simplify(16777216)
    snapshot = render_utils.render_snapshot(candidate_mesh, resolution=384, nviews=4)
    score = _silhouette_score(snapshot['alpha'])
    car_candidate_results.append({
        'seed': seed,
        'score': score,
        'frames': [frame.copy() for frame in snapshot['shaded']],
    })
    del candidate_mesh, snapshot
    gc.collect(); torch.cuda.empty_cache()

car_candidate_results.sort(key=lambda item: item['score'], reverse=True)
CAR_AUTO_SEED = car_candidate_results[0]['seed']
fig, axes = plt.subplots(len(car_candidate_results), 4, figsize=(14, 3.4 * len(car_candidate_results)), squeeze=False)
for row, result in enumerate(car_candidate_results):
    for col, frame in enumerate(result['frames']):
        axes[row, col].imshow(frame)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(f"seed {result['seed']}  |  score {result['score']:.3f}")
plt.tight_layout(); plt.show()
print('Automatic selection:', CAR_AUTO_SEED)
print('Use visual judgment: reject fused doors, duplicated wheels, missing roof sections, or large floating pieces.')

In [ ]:
#@title 19. Generate the selected high-quality Camry asset
CAR_SEED = -1  #@param {type:"integer"}
CAR_FINAL_MODE = 'stochastic'  #@param ["stochastic", "multidiffusion"]
CAR_FINAL_RESOLUTION = '1024_cascade'  #@param ["512", "1024", "1024_cascade", "1536_cascade"]
CAR_OUTPUT_NAME = 'open_door_camry_trellis2'  #@param {type:"string"}
# CAR_SEED=-1 uses cell 18's automatic winner. Enter a displayed seed to override it.
# 1024_cascade is the quality/stability default. Try 1536_cascade only after geometry is good.
# multidiffusion evaluates every selected view at each denoising step; runtime grows
# roughly with the number of images.

import os, torch
if CAR_SEED == -1:
    if 'CAR_AUTO_SEED' not in globals():
        raise RuntimeError('Run cell 18 first, or enter an explicit CAR_SEED.')
    selected_car_seed = CAR_AUTO_SEED
else:
    selected_car_seed = CAR_SEED
assert CAR_OUTPUT_NAME and os.path.basename(CAR_OUTPUT_NAME) == CAR_OUTPUT_NAME

car_mesh = pipeline.run_multi_image(
    car_images,
    seed=selected_car_seed,
    mode=CAR_FINAL_MODE,
    pipeline_type=CAR_FINAL_RESOLUTION,
    preprocess_image=False,
)[0]
car_mesh.simplify(16777216)
torch.cuda.empty_cache()
print(
    f'✓ high-quality car mesh generated with seed {selected_car_seed}: '
    f'{car_mesh.vertices.shape[0]:,} vertices, {car_mesh.faces.shape[0]:,} faces'
)

In [ ]:
#@title 20. Render the selected Camry turntable
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
from IPython.display import Video, display

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)
if car_mesh.device.type != 'cuda':
    car_mesh = car_mesh.cuda()
if 'envmap' not in globals():
    HDRI_PATH = 'assets/hdri/forest.exr'
    hdri = cv2.imread(HDRI_PATH, cv2.IMREAD_UNCHANGED)
    if hdri is None:
        raise FileNotFoundError(f'Could not read {HDRI_PATH}; re-run setup cell 7.')
    envmap = EnvMap(torch.tensor(
        cv2.cvtColor(hdri, cv2.COLOR_BGR2RGB), dtype=torch.float32, device='cuda'
    ))
car_video = render_utils.make_pbr_vis_frames(
    render_utils.render_video(car_mesh, envmap=envmap)
)
car_video_path = f'{OUT_DIR}/{CAR_OUTPUT_NAME}.mp4'
imageio.mimsave(car_video_path, car_video, fps=15)
display(Video(car_video_path, embed=True, width=512))

In [ ]:
#@title 21. Export the selected Camry GLB
CAR_EXPORT_REMESH = False  #@param {type:"boolean"}
CAR_DECIMATION_TARGET = 1000000  #@param {type:"integer"}
CAR_TEXTURE_SIZE = 4096  #@param [2048, 4096, 8192] {type:"raw"}

import gc, os, shutil, torch
import o_voxel
OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)
if car_mesh.device.type != 'cuda':
    car_mesh = car_mesh.cuda()
car_glb = o_voxel.postprocess.to_glb(
    vertices          = car_mesh.vertices,
    faces             = car_mesh.faces,
    attr_volume       = car_mesh.attrs,
    coords            = car_mesh.coords,
    attr_layout       = car_mesh.layout,
    voxel_size        = car_mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = CAR_DECIMATION_TARGET,
    texture_size      = CAR_TEXTURE_SIZE,
    remesh            = CAR_EXPORT_REMESH,
    remesh_band       = 1.0,
    remesh_project    = 0.9,
    verbose           = True,
)
car_glb_path = f'{OUT_DIR}/{CAR_OUTPUT_NAME}.glb'
car_glb.export(car_glb_path)
print('✓ exported', car_glb_path)
if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(car_glb_path, drive_out)
    if 'car_video_path' in globals() and os.path.exists(car_video_path):
        shutil.copy(car_video_path, drive_out)
    print('✓ copied GLB and turntable to', drive_out)
from google.colab import files as colab_files
colab_files.download(car_glb_path)
car_mesh = car_mesh.cpu()
for _name in ('car_glb', 'car_video'):
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

## Troubleshooting

| Symptom | Fix |
|---|---|
| Runtime verification fails | Choose **A100** and **Runtime Version 2025.10** under Runtime → Change runtime type, then reconnect. |
| `detected CUDA version … mismatches` during a build | A 12.5/12.6 warning is expected; a major-version error means the wrong Colab runtime is selected. |
| Import error / `undefined symbol` from a cached extension | Delete `Drive/TRELLIS2_cache/wheels/` and re-run cell 6. |
| `401/403` when downloading models | Accept the licenses for `facebook/dinov3-vitl16-pretrain-lvd1689m` and `briaai/RMBG-2.0` on Hugging Face, and check your `HF_TOKEN` Colab secret. |
| Multi-view upload is rejected | Cell 8 accepts any positive number of supported images. Check that at least one file has a PNG, JPG/JPEG, WebP, HEIC, or HEIF extension. |
| HEIC/HEIF image is unreadable | Rerun cell 4 to install `pillow-heif`, then cell 7 to register its Pillow decoder before uploading again. |
| CUDA OOM during generation | Set `RESOLUTION='512'` in cell 10 or `SINGLE_RESOLUTION='512'` in cell 14 and re-run that demo. |
| O-Voxel fails with `Eigen/Dense` or `Eigen/Core` not found | Rerun cell 6; it installs `libeigen3-dev` and links the headers into `o-voxel/third_party/eigen/`. |
| `Could not read assets/hdri/forest.exr` | Re-run cell 7 — the render cells read the HDRI relative to `/content/TRELLIS.2`. |
| Build fails on `git clone` of a pinned repo | Transient network error; just re-run cell 6. Successful wheels are already cached. |
| FlashAttention wheel is rejected | Confirm cell 1 reports Python 3.12, PyTorch 2.8, and runtime 2025.10; the notebook uses the matching official wheel. |
| A GLB has disconnected shards | Keep `EXPORT_REMESH=False` in cell 12 or `SINGLE_EXPORT_REMESH=False` in cell 16 so O-Voxel runs its full topology-cleanup path. |

**Re-running on the same session**: rerun cells 8→12 for a new multi-view set, or cells
13→16 for a new single front image. The pipeline stays loaded. Cells 12 and 16 park their
mesh on CPU when they finish so the other demo gets a clean GPU; re-running the matching
render or export cell moves it back automatically, so you only need to re-run generation
when you change an input or a generation parameter.